In [12]:
def get_ndvi_area(lat, lon, start_date, end_date):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    
    # 1. Cargar colección
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(region) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    
    # 2. Verificar si hay imágenes
    count = s2.size().getInfo()
    if count == 0:
        return "No hay imágenes disponibles para este periodo y filtro de nubes"
    
    # 3. Calcular NDVI
    def add_ndvi(image):
        return image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # 4. Procesar y reducir
    ndvi_col = s2.map(add_ndvi)
    mean_ndvi = ndvi_col.mean()
    
    stats = mean_ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=10,
        bestEffort=True
    )
    
    # 5. Extracción segura
    # Convertimos a diccionario y verificamos si la llave existe
    result = stats.getInfo()
    return result.get('NDVI', 'No se pudo calcular el promedio de NDVI')

In [13]:
start_date = '2025-01-01'
end_date = '2026-06-27'
ndvi_area = get_ndvi_area(7.3297, -73.1867, start, end)

print(ndvi_area)

No hay imágenes disponibles para este periodo y filtro de nubes


In [14]:
def check_image_inventory(lat, lon, start_date, end_date):
    point = ee.Geometry.Point([lon, lat])
    # Colección completa sin filtros restrictivos
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(point) \
        .filterDate(start_date, end_date)
    
    # Obtener lista de IDs y fechas
    count = s2.size().getInfo()
    if count > 0:
        # Obtenemos los IDs de las primeras 5 imágenes para inspeccionar
        sample = s2.limit(5).reduceColumns(ee.Reducer.toList(), ['system:time_start']).get('list').getInfo()
        return f"Imágenes encontradas: {count}. Fechas (timestamp): {sample}"
    else:
        return "No hay ninguna imagen de Sentinel-2 en esta ubicación para este rango de fechas."

# Ejecutar diagnóstico
print(check_image_inventory(7.3297, -73.1867, '2025-01-01', '2026-06-27'))

Imágenes encontradas: 297. Fechas (timestamp): [1735831857947, 1735831854548, 1736263858458, 1736263855070, 1736695855432]


In [20]:
def get_ndvi_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            # Definir meses del trimestre
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            # Filtrar imágenes Sentinel-2
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            # Función NDVI
            def add_ndvi(image):
                return image.normalizedDifference(['B8', 'B4']).rename('NDVI')
            
            # Calcular mediana trimestral
            ndvi_med = s2.map(add_ndvi).median()
            
            # Extraer valor promedio del área
            stats = ndvi_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).get('NDVI')
            
            # Guardar resultado (usando .getInfo() con manejo de errores)
            try:
                val = stats.getInfo()
                report[str(year)][f"Q{q}"] = val if val is not None else 0
            except:
                report[str(year)][f"Q{q}"] = 0
                
    return report

# Ejecutar con el rango donde sabemos que hay datos
start = 2020
end = 2026
ndvi_val = get_ndvi_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(ndvi_val, indent=4))

{
    "2020": {
        "Q1": 0.6806555782301085,
        "Q2": 0.2434579682190943,
        "Q3": 0.28592839296050193,
        "Q4": 0.7350524443844915
    },
    "2021": {
        "Q1": 0.7254982630691623,
        "Q2": 0.26414197976614406,
        "Q3": 0.3543036844509362,
        "Q4": 0.2447838506395563
    },
    "2022": {
        "Q1": 0.7604506734165835,
        "Q2": 0.6022565879057842,
        "Q3": 0.7042222677657237,
        "Q4": 0.3749842051241771
    },
    "2023": {
        "Q1": 0.7224360223200426,
        "Q2": 0.4734144660268068,
        "Q3": 0.42856136261556965,
        "Q4": 0.43149771115421337
    },
    "2024": {
        "Q1": 0.6664719453058198,
        "Q2": 0.14026554493961726,
        "Q3": 0.6848189316985139,
        "Q4": 0.538541972326514
    },
    "2025": {
        "Q1": 0.7871039634186136,
        "Q2": 0.33324353085536706,
        "Q3": 0.7020437941928293,
        "Q4": 0.7473026431385718
    },
    "2026": {
        "Q1": 0.2923392609015586,
        "

In [21]:
def get_vegetation_indices_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            def add_indices(image):
                ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
                ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
                ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
                # EVI requiere bandas B8(NIR), B4(Red), B2(Blue)
                evi = image.expression(
                    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
                        'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')
                    }).rename('EVI')
                # BSI requiere B4(Red), B11(SWIR1), B8(NIR), B2(Blue)
                bsi = image.expression(
                    '((RED + SWIR1) - (NIR + BLUE)) / ((RED + SWIR1) + (NIR + BLUE))', {
                        'RED': image.select('B4'), 'SWIR1': image.select('B11'),
                        'NIR': image.select('B8'), 'BLUE': image.select('B2')
                    }).rename('BSI')
                return image.addBands([ndvi, ndre, ndmi, evi, bsi])
            
            indices_med = s2.map(add_indices).median()
            
            stats = indices_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).getInfo()
            
            report[str(year)][f"Q{q}"] = stats
                
    return report

In [22]:
import json

# Define los parámetros
lat = 7.3297
lon = -73.1867
start = 2025
end = 2026

# Llama a la función
vegetation_data = get_vegetation_indices_by_year_and_quarter(lat, lon, start, end)

# Imprime el resultado de forma legible
print(json.dumps(vegetation_data, indent=4))

{
    "2025": {
        "Q1": {
            "AOT": 184.8765705953526,
            "B1": 410.8206806688792,
            "B11": 1742.2970403003126,
            "B12": 847.8418080848816,
            "B2": 376.18054478329645,
            "B3": 564.7437563987216,
            "B4": 355.533661154717,
            "B5": 991.4508578165226,
            "B6": 2565.938168957278,
            "B7": 3089.5956783420697,
            "B8": 3108.489467316102,
            "B8A": 3393.4015139763574,
            "B9": 3247.875903577077,
            "BSI": -0.23438068488406752,
            "EVI": 2.3519157393981565,
            "MSK_CLASSI_CIRRUS": 0,
            "MSK_CLASSI_OPAQUE": 0,
            "MSK_CLASSI_SNOW_ICE": 0,
            "MSK_CLDPRB": 0,
            "MSK_SNWPRB": 0,
            "NDMI": 0.2706133596155528,
            "NDRE": 0.5059245569543677,
            "NDVI": 0.7871039634186136,
            "QA10": null,
            "QA20": null,
            "QA60": 0,
            "SCL": 4,
            "TC

In [23]:
def get_clean_vegetation_indices(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}
    
    # Lista de índices que queremos
    indices_to_keep = ['NDVI', 'EVI', 'NDMI', 'BSI', 'NDRE']

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            def add_indices(image):
                ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
                ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
                ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
                evi = image.expression(
                    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
                        'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')
                    }).rename('EVI')
                bsi = image.expression(
                    '((RED + SWIR1) - (NIR + BLUE)) / ((RED + SWIR1) + (NIR + BLUE))', {
                        'RED': image.select('B4'), 'SWIR1': image.select('B11'),
                        'NIR': image.select('B8'), 'BLUE': image.select('B2')
                    }).rename('BSI')
                
                # Retornamos solo los índices calculados
                return image.addBands([ndvi, ndre, ndmi, evi, bsi]).select(indices_to_keep)
            
            # Calculamos la mediana de la colección filtrada
            indices_med = s2.map(add_indices).median()
            
            # Reducción espacial
            stats = indices_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).getInfo()
            
            # Asignar resultados (si stats está vacío, llenamos con 0)
            report[str(year)][f"Q{q}"] = {k: stats.get(k, 0) for k in indices_to_keep}
                
    return report

In [ ]:
import json

# Define los parámetros
lat = 7.3297
lon = -73.1867
start = 2020
end = 2026

# Llama a la función
vegetation_data = get_clean_vegetation_indices(lat, lon, start, end)

# Imprime el resultado de forma legible
print(json.dumps(vegetation_data, indent=4))

{
    "2020": {
        "Q1": {
            "NDVI": 0.6806555782301085,
            "EVI": 2.2475156913792005,
            "NDMI": 0.25384318953125085,
            "BSI": -0.2107604319829029,
            "NDRE": 0.47402390402372196
        },
        "Q2": {
            "NDVI": 0.2434579682190943,
            "EVI": -0.10251998745838169,
            "NDMI": 0.2706858740234612,
            "BSI": -0.184749234824521,
            "NDRE": 0.17766225262165755
        },
        "Q3": {
            "NDVI": 0.28592839296050193,
            "EVI": 0.3377652583949619,
            "NDMI": 0.27642764429351824,
            "BSI": -0.20033816116354994,
            "NDRE": 0.17847313344505414
        },
        "Q4": {
            "NDVI": 0.7350524443844915,
            "EVI": 2.288858350318132,
            "NDMI": 0.2636538799656235,
            "BSI": -0.20899167499232713,
            "NDRE": 0.4999164914582227
        }
    },
    "2021": {
        "Q1": {
            "NDVI": 0.7254982630691623,


In [7]:
import ee
from datetime import datetime, timedelta

def mask_s2_clouds(image):
    """
    Máscara de nubes y cirros para Sentinel-2 usando la banda QA60.
    """
    qa = image.select('QA60')

    # Bits 10 y 11 son nubes y cirros respectivamente
    cloud_bit_mask = 10
    cirrus_bit_mask = 11

    # Bits que deben ser 0 para que el píxel sea válido
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    # Aplicar la máscara y escalar los valores de reflectancia (de 0-10000 a 0-1)
    return image.updateMask(mask).divide(10000)

def get_sentinel2_ndvi_stats(lat, lon, buffer_meters=118):
    """
    Calcula el NDVI medio en un área de ~1 ha (radio ~118m) 
    utilizando Sentinel-2 para los últimos 6 meses.
    
    Args:
        lat (float): Latitud.
        lon (float): Longitud.
        buffer_meters (float): Radio del buffer (118m aprox 1 hectá$\\text{áre}$).
        
    Returns:
        float: Valor medio de NDVI, o None si no hay imágenes disponibles.
    """
    try:
        # 1. Definir punto y área de interés (ROI)
        point = ee.Geometry.Point([lon, lat])
        roi = point.buffer(buffer_meters)

        # 2. Definir rango de fechas (últimos 180 días)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=180)
        
        start_date_str = start_date.strftime('%Y-%m-%d')
        end_date_str = end_date.strftime('%Y-%m-%d')

        # 3. Cargar y filtrar colección Sentinel-2 L2A (Surface Reflectance)
        # Usamos la colección HARMONIZED para consistencia temporal
        s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(roi)
                  .filterDate(start_date_str, end_date_str)
                  # Filtro previo por metadatos: solo imágenes con < 20% de nubes
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                  .map(mask_s2_clouds))

        # Verificar si hay imágenes en la colección
        count = s2_col.size().getInfo()
        if count == 0:
            print(f"No se encontraron imágenes para el periodo {start_date_str} a {end_date_str}")
            return None

        # 4. Crear un compuesto de Mediana (Median Composite)
        # La mediana es robusta frente a nubes residuales y sombras
        composite = s2_col.median().clip(roi)

        # 5. Calcular NDVI: (B8 - B4) / (B8 + B4)
        # B8 = NIR (Near Infrared), B4 = Red
        ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

        # 6. Reducción espacial: Calcular la media de todos los píxeles en el ROI
        stats = ndvi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=10,  # Resolución nativa de Sentinel-2 (10m)
            maxPixels=1e9
        )

# ----

        try:
            # 2. ¡EL PASO CLAVE! Convertir el objeto del servidor a un diccionario de Python
            # .getInfo() descarga los datos y los convierte en un dict de Python real
            stats_dict = stats.getInfo() 
            
            # 3. Ahora stats_dict es un dict de Python, podemos usar .get() con seguridad
            # El nombre de la clave será el nombre de la banda (ej: 'ndvi')
            # Buscamos 'ndvi' o cualquier clave que contenga 'ndvi'
            value = None
            for key in stats_dict.keys():
                if 'ndvi' in key.lower():
                    value = stats_dict[key]
                    break
            
            return value

        except Exception as e:
            print(f"Error al extraer valor: {e}")
            return None

# ----
        # return stats.get('nd').getInfo() if 'nd' in stats else stats.get('ndvi', None)
        # Nota: El nombre de la banda puede variar según el proceso, 
        # pero tras el proceso de reduccion suele ser 'nd' o 'ndvi'
        
    except Exception as e:
        print(f"Error en el procesamiento: {e}")
        return None


In [8]:
# --- Ejemplo de Uso ---
if __name__ == "__main__":
    # Tu código aquí
    lat_test, lon_test = 7.3297, -71.1867 # ejemplo
    resultado = get_sentinel2_ndvi_stats(lat_test, lon_test)
    print(resultado)

0.7566806561540469


In [ ]:
## MDS650_v260805_terrain_profile_area

"""
Obtiene elevación, pendiente (slope) y orientación (aspect) promedio
para un área alrededor de un punto, usando SRTM o Copernicus DEM vía
Google Earth Engine.

CÓMO LEER EL RESULTADO
-----------------------
elevation_m       Elevación promedio del área, en metros sobre el nivel del mar.
elevation_std_m   Cuánto varía la elevación dentro del área. Un valor alto
                   indica terreno con desniveles marcados (no todo a la misma altura).

slope_deg         Pendiente promedio, en grados (0° = plano, 90° = vertical).
                   Como referencia general (no una regla estricta, revisa
                   siempre criterios agronómicos específicos para tu cultivo):
                     0-8°   : plano a suave, fácil manejo
                     8-15°  : moderado, puede requerir prácticas de conservación
                     15-25° : pronunciado, suele necesitar curvas de nivel/terrazas
                     >25°   : muy pronunciado, manejo más difícil/costoso

slope_std_deg     Cuánto varía la pendiente dentro del área. Alto = terreno
                   irregular (mezcla de zonas planas y empinadas); bajo =
                   pendiente consistente en toda el área (el promedio la
                   representa bien).

aspect_deg        Orientación hacia la que "mira" la pendiente, en grados
                   de brújula (0°=Norte, 90°=Este, 180°=Sur, 270°=Oeste).
                   Relevante para exposición solar: en el hemisferio norte,
                   laderas orientadas al sur/suroeste reciben más sol directo.
"""
import math
from datetime import datetime
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()


DEM_SOURCES = {
    "copernicus": "COPERNICUS/DEM/GLO30_2024_1",  # versión 2024, recomendado
    "srtm": "USGS/SRTMGL1_003",                    # clásico, puede tener huecos en zonas escarpadas
}


def get_terrain_profile_area(lat, lon, area_meters=56, dem_source="copernicus", verbose=False):
    """
    Calcula elevación, pendiente y orientación promedio sobre un área
    alrededor del punto. area_meters es el radio del buffer; el área
    real evaluada es un cuadrado de lado 2*area_meters (ver nota en
    save_terrain_profile / la conversación de diseño del pipeline).

    dem_source: "copernicus" (default, recomendado) o "srtm"
    verbose: si True, imprime las estadísticas crudas devueltas por GEE (debug)
    """
    if dem_source not in DEM_SOURCES:
        raise ValueError(f"dem_source debe ser uno de {list(DEM_SOURCES.keys())}")

    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        # COPERNICUS/DEM/GLO30 es una ImageCollection de tiles -> hay que mosaiquear
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')

    # IMPORTANTE: .mosaic() no conserva una proyección/escala de pixel bien
    # definida -> ee.Terrain.slope()/aspect() necesitan una grilla explícita
    # para calcular el gradiente correctamente. Sin esto, terminan usando
    # una escala por defecto mucho más gruesa que 30m.
    elevation_for_terrain = elevation.reproject(crs='EPSG:4326', scale=30)
    slope = ee.Terrain.slope(elevation_for_terrain).rename('slope')
    aspect_deg = ee.Terrain.aspect(elevation_for_terrain)

    # Aspect es un dato circular (0-360°) -> promediamos via seno/coseno,
    # no con una media aritmética directa (eso daría resultados incorrectos
    # cuando el área cruza el norte, ej. valores cerca de 0° y 360°).
    aspect_rad = aspect_deg.multiply(math.pi / 180)
    aspect_sin = aspect_rad.sin().rename('aspect_sin')
    aspect_cos = aspect_rad.cos().rename('aspect_cos')

    combined = elevation.addBands(slope).addBands(aspect_sin).addBands(aspect_cos)

    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.count(), sharedInputs=True)
    )

    stats = combined.reduceRegion(
        reducer=combined_reducer,
        geometry=region,
        scale=30,
        bestEffort=True
    ).getInfo()

    if verbose:
        print(f"[DEBUG] stats crudos: {stats}")

    mean_aspect_rad = math.atan2(stats['aspect_sin_mean'], stats['aspect_cos_mean'])
    mean_aspect_deg = math.degrees(mean_aspect_rad)
    if mean_aspect_deg < 0:
        mean_aspect_deg += 360

    results = {
        'lat': lat,
        'lon': lon,
        'dem_source': dem_source,
        'elevation_m': stats['elevation_mean'],
        'elevation_std_m': stats['elevation_stdDev'],
        'slope_deg': stats['slope_mean'],
        'slope_std_deg': stats['slope_stdDev'],
        'aspect_deg': mean_aspect_deg,
        # Nota: no calculamos un "std" de aspect por ser un dato circular
        # (requeriría varianza circular en vez de stdDev normal).
    }
    return results


def save_terrain_profile(terrain_data, out_prefix="terrain_profile_data", output_dir="../databases"):
    """
    Guarda el perfil de terreno con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame([terrain_data])

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    ha = 2
    area_meters = math.sqrt(ha * 10000) / 2
    Latitude=7.4584221918243045
    Longitude=-73.222052853104
    terrain_data = get_terrain_profile_area(Latitude, Longitude, area_meters, dem_source="copernicus")
    out_path, df = save_terrain_profile(terrain_data)
    print(df)

In [2]:
"""
Extrae la serie quincenal de indices de vegetacion (NDVI, EVI, NDMI,
BSI, NDRE) para un punto, usando Sentinel-2 Surface Reflectance
(COPERNICUS/S2_SR_HARMONIZED), con enmascarado de nubes via la banda
SCL (Scene Classification Layer).

POR QUE SCL Y NO QA60: QA60 (el bitmask de nubes "clasico") dejo de
producirse correctamente desde el 25-ene-2022 -> no es confiable para
toda la serie 2016-2026. SCL sigue generandose de forma consistente en
todo el periodo, por eso se usa como metodo principal de enmascarado.

LIMITACION IMPORTANTE DE FECHAS: COPERNICUS/S2_SR_HARMONIZED solo
tiene datos desde el 28-marzo-2017 (no desde 2016 como el resto del
pipeline) -> todas las quincenas de 2016 y enero-marzo de 2017 van a
salir en NaN, sin excepcion, no es un error del script. Ademas, ESA
advierte que la cobertura de 2017-2018 en la coleccion "no es todavia
global" -> puede haber huecos adicionales en esos dos primeros anos
incluso en zonas con cobertura teorica.

INDICES CALCULADOS (formulas estandar, bandas de reflectancia de
superficie escaladas a 0-1 antes de calcular):
    NDVI = (NIR - Red) / (NIR + Red)                     [B8, B4]
    EVI  = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)  [B8,B4,B2]
    NDMI = (NIR - SWIR1) / (NIR + SWIR1)                 [B8, B11]
    BSI  = ((SWIR1+Red) - (NIR+Blue)) / ((SWIR1+Red) + (NIR+Blue))
    NDRE = (NIR - RedEdge1) / (NIR + RedEdge1)           [B8, B5]

CITAS:
NDVI: Rouse, J.W., et al. (1974). Monitoring vegetation systems in the
    Great Plains with ERTS. NASA SP-351, 309-317.
EVI: Huete, A., et al. (2002). Overview of the radiometric and
    biophysical performance of the MODIS vegetation indices. Remote
    Sensing of Environment, 83(1-2), 195-213.
NDMI: Gao, B.C. (1996). NDWI-A normalized difference water index for
    remote sensing of vegetation liquid water from space. Remote
    Sensing of Environment, 58(3), 257-266.
BSI: Rikimaru, A., Roy, P.S., & Miyatake, S. (2002). Tropical forest
    cover density mapping. Tropical Ecology, 43(1), 39-47.
NDRE: Barnes, E.M., et al. (2000). Coincident detection of crop water
    stress, nitrogen status and canopy density using ground-based
    multispectral data. Proc. 5th Intl. Conf. on Precision Agriculture.

CÓMO LEER EL RESULTADO
-----------------------
ndvi, evi, ndmi, bsi, ndre   Mediana de cada indice sobre las imagenes
                              sin nubes disponibles en esa quincena.
n_images                     Cuantas imagenes (post-filtro de nubes)
                              se usaron para la mediana de esa quincena.
                              Si es 0, las columnas de indices quedan
                              en NaN (ver aviso impreso).
"""
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()
from test_period_utils import build_biweekly_periods


# SCL classes to exclude: 3=cloud shadow, 8=cloud medium prob,
# 9=cloud high prob, 10=cirrus, 11=snow/ice
SCL_CLOUD_CLASSES = [3, 8, 9, 10, 11]

INDEX_NAMES = ['ndvi', 'evi', 'ndmi', 'bsi', 'ndre']


def _mask_clouds_and_scale(image):
    """Enmascara nubes/sombras via SCL y escala reflectancia a 0-1."""
    scl = image.select('SCL')
    cloud_mask = scl.remap(SCL_CLOUD_CLASSES, [0] * len(SCL_CLOUD_CLASSES), 1)
    scaled = image.select(['B2', 'B3', 'B4', 'B5', 'B8', 'B11']).multiply(0.0001)
    return scaled.updateMask(cloud_mask).copyProperties(image, ['system:time_start'])


def _compute_indices(image):
    """Calcula los 5 indices de vegetacion a partir de bandas ya escaladas 0-1."""
    blue = image.select('B2')
    red = image.select('B4')
    red_edge1 = image.select('B5')
    nir = image.select('B8')
    swir1 = image.select('B11')

    ndvi = nir.subtract(red).divide(nir.add(red)).rename('ndvi')
    evi = nir.subtract(red).multiply(2.5).divide(
        nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)
    ).rename('evi')
    ndmi = nir.subtract(swir1).divide(nir.add(swir1)).rename('ndmi')
    bsi = swir1.add(red).subtract(nir.add(blue)).divide(
        swir1.add(red).add(nir).add(blue)
    ).rename('bsi')
    ndre = nir.subtract(red_edge1).divide(nir.add(red_edge1)).rename('ndre')

    result = ee.Image.cat([ndvi, evi, ndmi, bsi, ndre])
    # ee.Image.cat() combina bandas pero NO copia propiedades/metadatos
    # (como system:time_start) de las imagenes originales -> hay que
    # copiarlas explicitamente, o filterDate() no puede ubicar temporalmente
    # ninguna imagen resultante en ningun periodo
    return result.copyProperties(image, ['system:time_start'])


def get_sentinel2_indices_biweekly(lat, lon, start_date="2016-01-01", end_date=None,
                                    max_cloud_pct=80):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
                           (ver limitacion de fechas en el docstring del modulo)
    max_cloud_pct: pre-filtro a nivel de escena completa (rapido, poco
                   preciso); el enmascarado real pixel-a-pixel via SCL
                   se aplica despues, este solo descarta escenas
                   claramente inutilizables antes de procesarlas
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    s2_collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(point)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_pct))
        .filterDate(str(start), str(end + timedelta(days=1)))
        .map(_mask_clouds_and_scale)
        .map(_compute_indices)
    )

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = s2_collection.filterDate(p_start, p_end)
        n_images = filtered.size()

        # Mediana (mas robusta a residuos de nube/sombra que la media)
        composite = ee.Image(ee.Algorithms.If(
            n_images.gt(0),
            filtered.median(),
            ee.Image.constant([0] * len(INDEX_NAMES)).rename(INDEX_NAMES).selfMask()
        ))

        n_images_band = ee.Image.constant(n_images).toInt().rename('n_images')
        combined = composite.addBands(n_images_band)

        stats = combined.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=10,  # resolucion nativa de las bandas de 10m (B2,B3,B4,B8);
                       # B5/B11 (20m) se remuestrean automaticamente al reducir
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # unica llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        row = {
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            'lat': lat,
            'lon': lon,
            'n_images': props.get('n_images'),
        }
        for index_name in INDEX_NAMES:
            value = props.get(index_name)
            row[index_name] = round(value, 4) if value is not None else None
        rows.append(row)

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    missing_rows = df['ndvi'].isna().sum()
    if missing_rows > 0:
        missing_labels = df[df['ndvi'].isna()]['label'].tolist()
        print(f"[AVISO] {missing_rows} quincena(s) sin ninguna imagen sin nubes "
              f"disponible (incluye, como minimo, todo lo anterior a 2017-03-28 "
              f"por disponibilidad del dataset): {missing_labels}")

    return df


def save_sentinel2_indices_profile(df, out_prefix="sentinel2_indices_biweekly",
                                    output_dir="../databases"):
    """
    Guarda la serie de indices con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patron que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD

    Latitude, Longitude = 7.300921,              -73.009794

    # Reference points for quick access (commented out):
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    df = get_sentinel2_indices_biweekly(Latitude, Longitude, start_date="2016-01-01")
    out_path = save_sentinel2_indices_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-14
[AVISO] 103 quincena(s) sin ninguna imagen sin nubes disponible (incluye, como minimo, todo lo anterior a 2017-03-28 por disponibilidad del dataset): ['2016-01_Q1', '2016-01_Q2', '2016-02_Q1', '2016-02_Q2', '2016-03_Q1', '2016-03_Q2', '2016-04_Q1', '2016-04_Q2', '2016-05_Q1', '2016-05_Q2', '2016-06_Q1', '2016-06_Q2', '2016-07_Q1', '2016-07_Q2', '2016-08_Q1', '2016-08_Q2', '2016-09_Q1', '2016-09_Q2', '2016-10_Q1', '2016-10_Q2', '2016-11_Q1', '2016-11_Q2', '2016-12_Q1', '2016-12_Q2', '2017-01_Q1', '2017-01_Q2', '2017-02_Q1', '2017-02_Q2', '2017-03_Q1', '2017-03_Q2', '2017-04_Q1', '2017-04_Q2', '2017-05_Q1', '2017-05_Q2', '2017-06_Q1', '2017-06_Q2', '2017-07_Q1', '2017-07_Q2', '2017-08_Q1', '2017-08_Q2', '2017-09_Q1', '2017-09_Q2', '2017-10_Q1', '2017-10_Q2', '2017-11_Q1', '2017-11_Q2', '2017-12_Q1', '2017-12_Q2', '2018-01_Q1', '2018-01_Q2', '2018-02_Q1', '2018-02_Q2', '2018-03_Q1', '2018-03_Q2', '2018-04_Q1', '2018-04_Q2